# Part 4: Designing and training a round

A *round* is one batch of experiments that answers one question. This part covers the
two halves of a round: writing the design down before any GPU sees it, and running the
resulting jobs on the Nautilus cluster.

Round 14 is the worked example throughout. Its question: the Level-1 trigger will not
deliver the sixteen offline-style constituent features earlier rounds trained on. It
will deliver three. Does the thesis survive the trigger's real inputs?

## A variant is a config file and a seed

An experimental arm is fully specified by exactly two things: a JSON config file and an
integer seed. No environment variable to remember, no launch-time flag whose default
might differ between machines.

That matters because the config file then *is* the record. It is copied into the job,
echoed into the training log, and hashed into `train_meta.json` beside the checkpoint.
"What exactly was arm X?" has a one-file answer.

In [ ]:
import json

cfg = json.load(open("code/configs/r14-l1x3-n8-w1a8.json"))
print(json.dumps(cfg, indent=2))

Read that as three statements.

- **`arch`** — the round's fixed frame. For round 14 this is the round-8 `small`
  architecture unchanged (`d_model` 32, 2 layers, 4 heads, FFN 64, norm-free), except
  for the input contract: `n_part` and the three-feature `features` list.
- **`quant`** — the treatment. Here binary $\{-1,+1\}$ weights with 8-bit activations.
  The sibling configs differ *only* in `weight` and `act_bits`.
- **`train`** — the recipe, identical to round 8's down to the learning rate and the
  clipping mode, so no accuracy difference against the round-8 tables can be blamed on
  the optimizer.

## Configs are generated, never hand-edited

The configs live in `code/configs/`, one per arm, named `r<round>-<identifier>.json`,
and a per-round generator writes all of them. Round 14 has twenty configs produced by
two nested loops: one over the constituent count $N$, one over the quantization variant.

In [ ]:
import ast

src = open("code/configs/gen_r14.py").read()

print(ast.get_docstring(ast.parse(src)))    # the design, stated in the docstring
print("\n--- the two loops that define the arms ---")
for line in src.splitlines():
    if line.startswith(("N_SWEEP", "FEATURES", "VARIANTS", '    "fp32"', '    "w8a8"',
                        '    "w1a8"', '    "w1a6"', '    "w1a4"', "}")):
        print(line)

Every field outside those loops is held fixed **by construction**, across all twenty
files. Hand-edit one config and that guarantee is gone: two arms meant to differ only in
their treatment now differ in a second, unrecorded way, and any contrast between them is
confounded.

The generator is also the design in executable form. Its docstring says what varies,
what is held fixed, and why, and diffing two rounds' generators shows exactly what
changed between designs.

## Preflight: cheap insurance before GPU time

Between generation and launch stands a CPU-only preflight script, one per round. It must
print the literal marker `PREFLIGHT_ALL_PASS` before anything is submitted.

In [ ]:
# runs on: laptop (regenerate) | NRP CPU pod (preflight; no GPU burned)
!python code/configs/gen_r14.py     # rewrite all 20 round-14 configs
!./code/preflight_r13.sh            # round-13 pattern; must end PREFLIGHT_ALL_PASS

In [ ]:
for line in open("code/preflight_r13.sh"):
    if line.startswith("echo \"---") or "PREFLIGHT_ALL_PASS" in line:
        print(line.rstrip())

Each check turns an expensive late failure into a free early one.

1. **Build every config on CPU.** A mistyped key, a config/code version mismatch, or a
   broken import fails in seconds instead of forty minutes into a GPU job.
2. **Assert the exact parameter count**, no tolerance. The count is a cheap fingerprint
   of the architecture; the round-13 preflight asserts 19,201 for that round's
   deployable `small` stack, and a config that builds to another number is simply not
   the model the design registered. (Round 14's three-feature configs have their own
   count: 18,657 parameters for `r14-l1x3-n8-w1a8`, from
   `examples/export_verify_r14n8_w1a8s3.json`.)
3. **Check the binary gate.** For every arm claiming binary weights, each quantized
   layer's effective weights must take exactly two values, $\pm\beta$, with no zeros. A
   single zero would make the weights ternary, which is off-thesis, and would void the
   claim the project rests on.
4. **Probe the data URL** with an HTTP HEAD request: reachable, expected content length,
   nothing downloaded. Finding a data-host outage after forty jobs are queued costs a day.

Rounds add their own paranoia on top. The round-13 preflight carries a guarded-knob
regression proving the round's new training plumbing left every pre-existing config
byte-identical in behavior: new code had to be demonstrably inert for old designs before
it could be trusted with new ones.

## Pre-registration: the design is fixed and dated before launch

Round 14's design was written down on 2026-08-01, before any job existed.

**The change, and only this change.** Cut the feature set from sixteen to the three the
trigger provides ($p_T$, $\eta_{rel}$, $\phi_{rel}$), resolved by *exact suffix match* in
the loader so `pt` can never silently match `ptrel`, and applied only after the $p_T$
sort so constituent ordering is unchanged. Everything else is the round-8 recipe
verbatim, so the input set is the only variable against the round-7/8 tables and the
constituent count is the only variable within the round.

**The arms and seeds**, fixed in advance:

| stage | arms | $N$ | seeds | jobs |
|---|---|---|---|---|
| 1 | fp32, w8a8, w1a8 (baselines + thesis) | 8, 16, 32, 64 | 1, 2, 3 | 36 |
| 2 | w1a6, w1a4 (activation ladder) | 8, 16, 32, 64 | 1, 2, 3 | 24 |

**The selection rule**, also fixed in advance: within a run, the checkpoint is the epoch
with the best validation macro one-vs-rest AUC; the reported number is the ROC-test
macro-OvR AUC on the held-out 260,000-jet split, as seed mean ± sample standard
deviation; no cross-arm claim without an uncertainty interval (part 5).

**The falsifier**, stated before the data existed: if w1a8 sits within noise of the
baselines at some $N$, with zero DSPs downstream, the thesis holds at trigger-realistic
inputs. If the binary gap blows up on three-feature inputs, that is a result to report,
not a failure to explain away. A separate sanity gate keeps pathology out of that
judgment: a collapsed AUC near 0.5 means investigate and re-run, never publish.

Why insist on all this before launch? Because post-hoc selection is invisible in the
final table. Four values of $N$ times five variants is twenty places for an encouraging
fluctuation to appear, and "report the $N$ where binary looks best" is a garden of
forking paths that no downstream statistics repairs. Fixing the arms, the seeds, and the
reporting rule in advance removes the freedom to wander.

## Staged launches: a stage is a decision boundary

Stages are not a batching convenience. A stage boundary is where a pre-registered
decision gets made.

Round 13 is the clean example. Stage 0 was a learning-rate probe: four single-seed
configs spanning $2\times10^{-5}$ to $2\times10^{-4}$. The stage-1 production configs
carried a placeholder learning rate that stage 0 would replace; the winner is pinned by
regenerating the configs with `gen_r13.py --lr <winner>`, and the preflight prints an
advisory flag against any production config still carrying the placeholder. Launching
stage 1 early would mean either training at an unjustified rate or, worse, leaving room
to choose the rate after seeing production results.

In [ ]:
import re

txt = open("code/jobs/launch_r14.sh").read()
for stage in ("stage1", "stage2"):
    jobs = re.search(rf"^{stage}=\((.*?)\)$", txt, re.M).group(1).split()
    print(f"{stage}: {len(jobs)} jobs   e.g. {jobs[0]}")

## Three seeds, and what they are for

Every production arm runs three seeds, and every reported accuracy is the seed mean ±
sample standard deviation. Three is a floor, not a luxury: training variance is real and
uneven across arms.

In the FINAL campaign the binary arm's seed spread was ±0.0054 in ROC-test macro-OvR AUC
while the W8A8 baseline's was ±0.0007 (`bnjettag/roc-results/final/roc_auc.md`) — nearly
an order of magnitude apart, easily enough to manufacture a fake win or mask a real loss
from a single seed. So the corollary rule is absolute: no "best seed" ever appears in a
table. All three are reported, and the spread travels with the mean into every
comparison.

## The round tag

One tag, `r<N>`, propagates through every layer of a round: the generator, the configs,
the code bundle, the job manifests, the W&B group and tags, and the result stores. Any
artifact missing the tag was improvised outside the registered design, and improvised
artifacts do not enter the record.

In [ ]:
import json, re

cfg = json.load(open("code/configs/r14-l1x3-n8-w1a8.json"))
yaml = open("code/jobs/kai-bn14-l1x3-n8-w1a8-s1.yaml").read()

print("config name    :", cfg["name"])
print("job name       :", re.search(r"^  name: (\S+)", yaml, re.M).group(1))
for key in ("WANDB_PROJECT", "WANDB_GROUP", "WANDB_TAGS"):
    print(f"{key:15s}:", re.search(rf"{key}=(\S+)", yaml).group(1))

## Training runs as a batch Job, never an interactive pod

Interactive pods holding idle GPUs get reaped by cluster policy, and a campaign that
depends on a human keeping a shell alive is not reproducible. A `batch/v1` Job also buys
three properties the design needs:

- `backoffLimit: 2` tolerates the cluster's known node-admission race (a scheduling
  failure with zero side effects) without letting a genuinely broken job retry forever.
- `activeDeadlineSeconds: 21600` bounds any run to six hours, sized to the 101-epoch
  recipe at the longest sequence length with margin, so a hung job cannot hold a GPU for
  a weekend.
- `restartPolicy: Never` makes a retry a fresh pod, not a mutated survivor.

The Job's `.status.succeeded` / `.status.failed` fields give the launch script something
machine-readable to poll.

## Code ships as a ConfigMap

The pods mount no shared filesystem at all — a stance adopted after the cluster storage
outage of early July 2026 (part 8). Code therefore ships inside the cluster's own object
store, as a ConfigMap holding a compressed tarball of the training tree.

In [ ]:
# runs on: laptop (needs kubectl access to the cluster)
!./code/jobs/make_code_configmap_r14.sh

The builder excludes caches and checkpoints to stay under the 1 MB ConfigMap ceiling,
refuses to publish a tree that would fail the in-pod staleness check, and prints the
tarball's md5 — which every pod echoes into its own log, so any log can be matched to the
exact code bytes it ran.

There is one ConfigMap per round, and once a round is launched its ConfigMap is frozen.
Freezing serves reproducibility (re-running an old round means re-running *its* code),
but it is also a safety rule with a scar behind it: refreshing a ConfigMap under running
pods killed eleven live jobs on 2026-07-25 (part 8).

## Anatomy of a generated job

Job manifests are generated, never hand-written, by a per-round script
(`code/jobs/gen_r14_jobs.py`) holding one template and a stage table of (config, seed)
pairs. It refuses to emit anything if a config file is missing or a job name exceeds
Kubernetes' 63-character limit.

The helper below prints slices of one real generated manifest, block by block.

In [ ]:
LINES = open("code/jobs/kai-bn14-l1x3-n8-w1a8-s1.yaml").read().splitlines()

def show(start, end, label=""):
    if label:
        print(f"# --- {label}  (lines {start}-{end}) ---")
    for i in range(start, end + 1):
        print(f"{i:4d}  {LINES[i - 1]}")

In [ ]:
show(24, 34, "scheduling: host blocklist + architecture")
print()
show(56, 64, "scheduling: which GPUs we prefer")

The blocklist names hosts that have previously eaten jobs; a single bad node consumed
seventeen on 2026-07-25 (part 8). After that the terms constrain to amd64 and an
explicit GPU allowlist (omitted above; it is the long `values:` list on lines 35-55).

The preference weights invert the usual instinct: weight 100 goes to the abundant
mid-tier (A10, L4, V100, A40, RTX 3090 class) and weight 10 to H100/H200. A model with
$d_{model}=32$ is bound by its data pipeline, not by FLOPs, so end-to-end time is
dominated by queue time — and the mid-tier queues are short. Flagship GPUs are a last
resort, not a prize.

In [ ]:
show(98, 114, "the stale-code gate")

This is the most important block in the file. The dangerous failure mode in this kind of
work is not a crash: an old code tree running a new config *works*. A pre-round-14 tree
would accept `r14-l1x3-n8-w1a8.json`, ignore the `features` key it does not know, and
train happily on all sixteen input features — the exact confound round 14 exists to
remove — while labeling the run as if it had not.

So before a single epoch runs, the job hard-fails unless the unpacked tree provably
contains the round's own markers: the round's config file, `feature_indices` in
`bnhgq2/data.py`, and the current artifact-upload layout in `bnhgq2/train.py`. The gate
converts a silent mislabeled experiment into a loud fatal error with a printed remedy.
It has already paid for itself: on 2026-07-25 a ConfigMap propagation race fed eleven
starting pods a stale tree, and they died here instead of training mislabeled runs.

In [ ]:
show(116, 121, "pinned dependencies")
print()
show(122, 132, "GPU assertion and the data gate")

The container is stock `python:3.12` and every dependency is pip-installed at an exact
pinned version. The `LD_LIBRARY_PATH` line is a workaround for the TensorFlow 2.21
wheel's broken CUDA-library discovery, verified 2026-07-07.

Then two fail-fast checks. `nvidia-smi` must succeed *and* TensorFlow must report a
visible GPU, or the job dies immediately — failing in thirty seconds beats discovering ten hours
later that the run trained on CPU.

Then each pod downloads the training tarball from Zenodo itself, every run, and gates on
its size: the transfer must exceed 2.5 GB against the known true size of 2,725,115,104
bytes, measured 2026-07-07. Without the gate, a truncated download looks like a small
dataset rather than an error.

In [ ]:
show(133, 143, "the environment contract")
print()
show(146, 151, "the training command")

`BNHGQ2_TRAIN_DATA` is set only *after* the download passed its size gate, so the trainer
can treat the path as trusted. The W&B block is the round tag made concrete: run name,
group, and tags all carry `r14`, and round 14 writes into its own W&B project precisely
so its 3-feature numbers can never sit unlabeled next to 16-feature ones. The API key is
injected from a Kubernetes secret, never from the YAML.

And the training command is the *only* entry point: `run_stage.py train --config --seed`
is the entire interface, with the log teed to a file that ships home beside the results.

## Launch waves and monitoring

The generator also writes the round's launch script. It deletes any prior job of the
same name, applies manifests in waves of six, and polls completion status every 120
seconds before releasing the next wave — sixty simultaneous submissions would mostly sit
in queue while starving other users. Stages run strictly in order.

In [ ]:
# runs on: laptop (kubectl against the cluster)
!./code/jobs/launch_r14.sh stage1
!kubectl get jobs,pods -n cms-ml
!kubectl logs -f kai-bn14-l1x3-n8-w1a8-s1-xxxxx -n cms-ml

Monitoring is unglamorous: `get jobs,pods` for scheduling state, `logs -f` to watch one
run pass its gates, and the campaign's W&B project for live loss and validation-AUC
curves.

Read the curves comparatively, not absolutely. The three seeds of an arm share a W&B
group, so a seed diverging from its siblings mid-training is visible days before any
test-set number exists, and an arm whose validation curve saturates suspiciously early
flags a recipe problem while the round can still be stopped cheaply. A run that passes
the GPU assertion, the data gate, and the first epoch almost always finishes: the failure
mass sits at the start, which is exactly where the gates are.

## What a run produces

Three files (plus the log) that together form the model's complete interface contract:

- **`model_best.keras`** — the checkpoint of the best epoch *by validation macro
  one-vs-rest AUC*. Validation AUC selects the epoch and drives early stopping; it is
  never the number of record (part 5).
- **`train_meta.json`** — variant, seed, config hash, parameter count, best epoch and its
  validation AUC, timestamps. Enough provenance to reconstruct what the checkpoint is
  without asking anyone.
- **`input_std.json`** — the per-feature standardization statistics, fitted on the
  training split only. This file is part of the model, not a convenience: a checkpoint
  scored without its own `input_std.json` is scored wrong, so it travels with the
  checkpoint everywhere.

Rounds trained under EBOPs pressure add a Pareto front, but those three are the
invariant core.

## Why the artifact upload is fatal

The pod's disk is an `emptyDir`: it evaporates the moment the pod terminates, successful
or not. Durability has exactly one path. Before exiting, the run uploads a versioned
artifact named `model-<leaf>` to W&B, and the upload *blocks* until the server confirms
the commit and the manifest checks out; for a real run, a failed upload fails the job.

That versioned artifact, not any file on any disk, is the checkpoint of record: the ROC
evaluation (part 5) and the hardware export (part 6) both fetch from it and from nowhere
else. The rule was written after 2026-07-15, when four of eight checkpoints from a
finished batch turned out to have been silently lost in upload (part 8). A training run
that cannot prove its checkpoint is durable has, for the record's purposes, not happened.